In [ ]:
from src.loaders.persona_loader import load_persona, build_timeline

p01 = load_persona("p01")
p05 = load_persona("p05")

sources = ["lifelog", "conversations", "emails", "calendar", "social_posts", "transactions", "files_index"]

print("p01 record counts:")
for s in sources:
    print(f"  {s}: {len(p01[s])}")

print("\np05 record counts:")
for s in sources:
    print(f"  {s}: {len(p05[s])}")


In [ ]:
jsonl_sources = ["lifelog", "conversations", "emails", "calendar", "social_posts", "transactions", "files_index"]

for s in jsonl_sources:
    df = p01[s]
    null_ts = int(df["ts"].isna().sum())
    min_ts = df["ts"].min()
    max_ts = df["ts"].max()
    status = "✅" if null_ts == 0 else "❌"
    print(f"{status} {s}: null_ts={null_ts}, min_ts={min_ts}, max_ts={max_ts}")


In [ ]:
from collections import Counter

lifelog_tags = Counter(tag for tags in p01["lifelog"]["tags"] for tag in tags)
conv_tags = Counter(tag for tags in p01["conversations"]["tags"] for tag in tags)

print("Top 20 lifelog tags:")
for tag, n in lifelog_tags.most_common(20):
    print(f"  {tag}: {n}")

print("\nTop 20 conversation tags:")
for tag, n in conv_tags.most_common(20):
    print(f"  {tag}: {n}")


In [ ]:
from src.features.stress_scorer import compute_stress
import plotly.express as px

stress = compute_stress(p01["calendar"])
fig = px.line(stress, x="date", y=["stress_raw", "stress_smooth"], title="p01 Daily Stress")
fig.show()


In [ ]:
from src.features.spend_tagger import tag_spend
import plotly.express as px

txn, weekly = tag_spend(p01["transactions"])
fig = px.bar(weekly, x="year_week", y="weekly_discretionary_total", title="p01 Weekly Discretionary Spend")
fig.show()


In [ ]:
stress_min = stress["date"].min()
stress_max = stress["date"].max()
txn_min = txn["date"].min()
txn_max = txn["date"].max()

print(f"Stress date range: {stress_min} -> {stress_max}")
print(f"Transactions date range: {txn_min} -> {txn_max}")

overlap = (stress_max >= txn_min) and (txn_max >= stress_min)
print("✅ Overlap exists" if overlap else "❌ No overlap")


In [ ]:
import random

all_ids = set()
for s in ["lifelog", "conversations", "emails", "calendar", "social_posts", "transactions", "files_index"]:
    if "id" in p01[s].columns:
        all_ids.update(p01[s]["id"].dropna().astype(str).tolist())

lifelog_refs_rows = p01["lifelog"][p01["lifelog"]["refs"].apply(lambda x: isinstance(x, list) and len(x) > 0)]
sample_n = min(10, len(lifelog_refs_rows))
sampled = lifelog_refs_rows.sample(sample_n, random_state=42) if sample_n > 0 else lifelog_refs_rows

resolved = 0
for refs in sampled["refs"]:
    if all(str(r) in all_ids for r in refs):
        resolved += 1

print(f"{resolved}/{sample_n} refs resolved")


In [ ]:
from src.insights.insight_engine import save_insights

save_insights("p01")
save_insights("p05")
print("✅ insights cache generated for p01 and p05")
